# LF Whale Hunter

Look up Polymarket proxy wallet addresses from a list of usernames,
fetch their trade history, and produce a `WhaleAnalysis` for each.

In [1]:
from src.trading_tools.apps.whale_monitor.analyser import analyse_markets

%load_ext autoreload
%autoreload 2

import logging
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path("../../..").resolve() / ".env")

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)
print("Setup complete.")

Setup complete.


/Users/oliverwarwick/playground/trading-tools/.venv/lib/python3.14/site-packages/pandera/_pandas_deprecated.py:157: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


## Configuration

In [2]:
INPUT_CSV = Path("whales_to_watch/lf_whales_input.csv")
MAX_TRADE_PAGES = 10  # pages of 500 trades per wallet (increase for deeper history)
ENRICH = True  # fetch Gamma market metadata for each trade (cached per market)

## Step 1 — Load usernames from CSV

In [3]:
users_df = pd.read_csv(INPUT_CSV)
usernames = users_df["username"].tolist()
print(f"Loaded {len(usernames)} usernames: {usernames}")

Loaded 5 usernames: ['IBOV200K', 'RememberAmalek', 'kch123', '0xd8f8c13644ea84d62e1ec88c5d1215e436eb0f11', 'xdd07070']


## Step 2 — Resolve usernames to proxy wallet addresses

Uses `PolymarketClient.lookup_user_address()` which queries the
`/v1/leaderboard?userName=` endpoint.

In [4]:
from trading_tools.clients.polymarket.client import PolymarketClient

# Use cached addresses from the CSV where available; fall back to API lookup.
address_map: dict[str, str | None] = {}
cached = dict(
    zip(users_df["username"], users_df.get("address", [None] * len(users_df)), strict=False)
)

to_lookup = [u for u, a in cached.items() if pd.isna(a) or not a]

if to_lookup:
    async with PolymarketClient() as client:
        for username in to_lookup:
            cached[username] = await client.lookup_user_address(username)

address_map = {u: (None if pd.isna(a) else str(a)) for u, a in cached.items()}

found = {u: a for u, a in address_map.items() if a}
missing = [u for u, a in address_map.items() if not a]

for u, a in address_map.items():
    print(f"  {u:20s} -> {a or 'NOT FOUND'}")
print(f"\nResolved {len(found)}/{len(usernames)} usernames.")
if missing:
    print(f"Not found: {missing}")

  IBOV200K             -> 0x8c0b024c17831a0dde038547b7e791ae6a0d7aa5
  RememberAmalek       -> 0x6139c42e48cf190e67a0a85d492413b499336b7a
  kch123               -> 0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee
  0xd8f8c13644ea84d62e1ec88c5d1215e436eb0f11 -> NOT FOUND
  xdd07070             -> 0x25e28169faea17421fcd4cc361f6436d1e449a09

Resolved 4/5 usernames.
Not found: ['0xd8f8c13644ea84d62e1ec88c5d1215e436eb0f11']


## Step 3 — Collect trade history

Fetches raw trades from `/trades?user=<address>` and converts each dict to a
`WhaleTrade` instance via `_parse_trade()`.

In [5]:
from trading_tools.apps.whale_monitor.collector import _parse_trade
from trading_tools.apps.whale_monitor.enricher import MarketMetadata, enrich_trades
from trading_tools.clients.polymarket.client import PolymarketClient

_TRADES_PER_PAGE = 500

# username -> list[WhaleTrade] (plain) or list[EnrichedTrade] (when ENRICH=True)
whale_trades: dict[str, list] = {}

# Shared enrichment cache — condition IDs resolved here are not re-fetched
# across traders, so the API is called at most once per unique market.
enrich_cache: dict[str, MarketMetadata] = {}

async with PolymarketClient() as client:
    for username, address in found.items():
        try:
            raw_trades = []
            now_ms = int(time.time() * 1000)

            for page_num in range(MAX_TRADE_PAGES):
                page = await client.get_trader_trades(
                    address,
                    limit=_TRADES_PER_PAGE,
                    offset=page_num * _TRADES_PER_PAGE,
                )
                raw_trades.extend(page)
                if len(page) < _TRADES_PER_PAGE:
                    break

            trades = [
                t for raw in raw_trades if (t := _parse_trade(raw, address, now_ms)) is not None
            ]

            if ENRICH:
                trades = await enrich_trades(client, trades, cache=enrich_cache)

            whale_trades[username] = trades
            print(f"{username}: {len(trades)} trades collected" + (" (enriched)" if ENRICH else ""))
        except Exception as e:
            logging.debug(f"Error fetching trades for {username} ({address}): {e}")

if ENRICH:
    print(f"\nEnrichment cache: {len(enrich_cache)} unique markets resolved")

xdd07070: 623 trades collected (enriched)

Enrichment cache: 49 unique markets resolved


## Step 4 — Analyse

Runs `analyse_trades()` over each entry in `whale_trades`.

In [6]:
from trading_tools.apps.whale_monitor.analyser import analyse_trades

trade_analysis = {
    username: analyse_trades(found[username], trades) for username, trades in whale_trades.items()
}

market_analysis = {username: analyse_markets(trades) for username, trades in whale_trades.items()}

for username, analysis in trade_analysis.items():
    print(
        f"{username}: {analysis.total_trades} trades, "
        f"volume=${analysis.total_volume:,.0f}, "
        f"{analysis.unique_markets} markets"
    )

xdd07070: 623 trades, volume=$945,906, 49 markets


## Step 5 — Inspect results

In [7]:
whale_trades["xdd07070"][0]

EnrichedTrade(trade=<trading_tools.apps.whale_monitor.models.WhaleTrade object at 0x10f18e510>, category='Esports', is_recurring=True, recurrence='daily', outcome_structure=<OutcomeStructure.MULTI: 'MULTI'>, close_datetime=datetime.datetime(2026, 3, 17, 19, 0, tzinfo=datetime.timezone.utc), winning_outcome=None, trade_pnl=None, is_active=True)

## Step 5 — summarize_trades + breakdown functions

In [8]:
from trading_tools.apps.whale_monitor.analyser import (
    hourly_distribution,
    market_breakdown,
    outcome_breakdown,
    side_breakdown,
    summarize_trades,
    trades_to_df,
)

# ── Scalar summary for each wallet ────────────────────────────────────────────
summaries = {
    username: summarize_trades(found[username], trades) for username, trades in whale_trades.items()
}

# Build a single multi-wallet summary DataFrame using to_series()
summary_df = pd.DataFrame([s.to_series() for s in summaries.values()])
display(summary_df)

# ── Breakdowns for a single wallet ────────────────────────────────────────────
username = next(iter(whale_trades))
df = trades_to_df(whale_trades[username])

print(f"\n--- {username}: market breakdown ---")
display(market_breakdown(df).head(10))

print(f"\n--- {username}: outcome breakdown ---")
display(outcome_breakdown(df))

print(f"\n--- {username}: side breakdown ---")
display(side_breakdown(df))

print(f"\n--- {username}: hourly distribution ---")
display(hourly_distribution(df))

,address,total_trades,total_volume,buy_count,sell_count,avg_size,avg_buy_price,avg_sell_price,unique_markets,avg_trade_duration_s,min_trade_duration_s,max_trade_duration_s,single_side_market_pct,average_daily_trades_1w,average_daily_trades_1m,sharpe_ratio
0,0x25e28169faea17421fcd4cc361f6436d1e449a09,623,945906.423563,604,19,1855.376454,0.751962,0.883316,49,44303.76,2892.0,231127.0,61.2,0.57,2.7,None



--- xdd07070: market breakdown ---


,title,count
0,Dota 2: Team Falcons vs Xtreme Gaming (BO3) - ...,116
1,LoL: Anyone's Legend vs Invictus Gaming (BO5) ...,70
2,LoL: Team WE vs ThunderTalk Gaming (BO3) - LPL...,54
3,LoL: Anyone's Legend vs Invictus Gaming (BO3) ...,42
4,LoL: Ninjas in Pyjamas vs ThunderTalk Gaming (...,41
5,LoL: Weibo Gaming vs Top Esports (BO3) - LPL G...,29
6,LoL: ThunderTalk Gaming vs EDward Gaming (BO3)...,22
7,LoL: ThunderTalk Gaming vs Ninjas in Pyjamas (...,21
8,LoL: Top Esports vs Anyone's Legend (BO3) - LP...,21
9,LoL: Team Heretics vs G2 Esports (BO1) - LEC V...,20



--- xdd07070: outcome breakdown ---


,outcome,count
0,Anyones Legend,135
1,Team Falcons,116
2,Ninjas in Pyjamas,62
3,Team WE,54
4,EDward Gaming,36
5,Weibo Gaming,29
6,LNG Esports,25
7,Bilibili Gaming,21
8,G2 Esports,20
9,PARIVISION,17



--- xdd07070: side breakdown ---


,side,count
0,BUY,604
1,SELL,19



--- xdd07070: hourly distribution ---


,hour,count
0,0,36
1,1,25
2,2,48
3,3,43
4,4,49
5,5,19
6,6,8
7,7,11
8,8,10
9,9,6


In [ ]:
trade_analysis["xdd07070"].market_breakdown